# Sleep Stage — Test Set Evaluation
Retrain XGBoost + LightGBM on **train + val**, then evaluate on the held-out **test** set.


In [ ]:
# ── Standard library ──────────────────────────────────────────────────────────
import importlib.util
import json
import sys
from collections import Counter
from pathlib import Path

# ── Third-party ───────────────────────────────────────────────────────────────
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from lightgbm import Booster, LGBMClassifier
from sklearn.metrics import classification_report, confusion_matrix, f1_score
from sklearn.utils.class_weight import compute_sample_weight
from xgboost import XGBClassifier

# ── Project source modules ─────────────────────────────────────────────────────
PROJECT_ROOT = Path("..").resolve()
SRC_DIR = PROJECT_ROOT / "src"

def _load_module(name, path):
    spec = importlib.util.spec_from_file_location(name, path)
    mod = importlib.util.module_from_spec(spec)
    sys.modules[name] = mod
    spec.loader.exec_module(mod)
    return mod

sleep_data_utils = _load_module("sleep_data_utils", SRC_DIR / "sleep_data_utils.py")
sleep_model_utils = _load_module("sleep_model_utils", SRC_DIR / "sleep_model_utils.py")

DEFAULT_STAGE_MAP          = sleep_data_utils.DEFAULT_STAGE_MAP
SleepDataset               = sleep_data_utils.SleepDataset
build_subject_list         = sleep_data_utils.build_subject_list
encode_labels              = sleep_data_utils.encode_labels
flatten_dataset            = sleep_data_utils.flatten_dataset
load_sleep_config          = sleep_data_utils.load_sleep_config
load_or_create_subject_split = sleep_data_utils.load_or_create_subject_split
render_subject_split_file_template = sleep_data_utils.render_subject_split_file_template

plot_confusion_matrix      = sleep_model_utils.plot_confusion_matrix
SleepFeaturePipeline       = sleep_model_utils.SleepFeaturePipeline
mode_filter_1d             = sleep_model_utils.mode_filter_1d

# ── Config ─────────────────────────────────────────────────────────────────────
CONFIG_PATH = PROJECT_ROOT / "config" / "sleep-model.yaml"
cfg = load_sleep_config(CONFIG_PATH)

SMALL_BATCH_TEST    = cfg["subjects"]["small_batch_test"]
FEAT_COLS           = cfg["data"]["feature_cols"]
TARGET_COL          = cfg["data"]["target_cols"]
FEAT_METADATA_COLS  = cfg["data"]["feature_metadata_cols"]
STAGE_MAP           = cfg.get("labels", {}).get("stage_map", DEFAULT_STAGE_MAP)

SIGNAL_DIR   = str((CONFIG_PATH.parent / cfg["data"]["signal_dir"]).resolve())
METADATA_CSV = str((CONFIG_PATH.parent / cfg["data"]["metadata_csv"]).resolve())

ACROSS_SUBJECT_RATIO   = cfg["split"]["across_subject_ratio"]
subject_split_template = cfg["split"].get(
    "subject_split_file",
    "../data/processed/splits/subject_split_seed{seed}_ratio_{ratio_tag}.json",
)
SPLIT_FILE = str((
    CONFIG_PATH.parent / render_subject_split_file_template(
        subject_split_template,
        seed=cfg["split"]["seed"],
        ratio=ACROSS_SUBJECT_RATIO,
    )
).resolve())

SUBJECTS = build_subject_list(
    prefix=cfg["subjects"]["prefix"],
    start=cfg["subjects"]["start"],
    end=cfg["subjects"]["end"],
    small_batch_test=SMALL_BATCH_TEST,
    small_batch_limit=cfg["subjects"]["small_batch_limit"],
)
print("Config loaded.")


## Load Dataset


In [ ]:
split_payload = load_or_create_subject_split(
    subject_ids=SUBJECTS,
    split_file_path=SPLIT_FILE,
    ratio=cfg["split"].get("across_subject_ratio", cfg["split"]["within_subject_ratio"]),
    seed=cfg["split"]["seed"],
)

train_subs = split_payload["train_subjects"]
val_subs   = split_payload["val_subjects"]
test_subs  = split_payload["test_subjects"]

print(f"Loaded split: {SPLIT_FILE}")
print(f"Subjects — train: {len(train_subs)}, val: {len(val_subs)}, test: {len(test_subs)}")

_ds_kwargs = dict(
    signals_dir=SIGNAL_DIR,
    metadata_csv=METADATA_CSV,
    feature_cols=FEAT_COLS,
    label_cols=TARGET_COL,
    block_duration_sec=cfg["dataset"]["block_duration_sec"],
    epoch_sec=cfg["dataset"]["epoch_sec"],
    drop_boundary=cfg["dataset"]["drop_boundary"],
    meta_feature_cols=FEAT_METADATA_COLS,
)

train_ds = SleepDataset(allowed_subjects=train_subs, **_ds_kwargs)
val_ds   = SleepDataset(allowed_subjects=val_subs,   **_ds_kwargs)
test_ds  = SleepDataset(allowed_subjects=test_subs,  **_ds_kwargs)

X_train, y_train, info_train = flatten_dataset(train_ds)
X_val,   y_val,   info_val   = flatten_dataset(val_ds)
X_test,  y_test,  info_test  = flatten_dataset(test_ds)

X_train, y_train, info_train = encode_labels(X_train, y_train, info_train, stage_map=STAGE_MAP)
X_val,   y_val,   info_val   = encode_labels(X_val,   y_val,   info_val,   stage_map=STAGE_MAP)
X_test,  y_test,  info_test  = encode_labels(X_test,  y_test,  info_test,  stage_map=STAGE_MAP)

print(f"X_train: {X_train.shape}  X_val: {X_val.shape}  X_test: {X_test.shape}")


## Feature Engineering Pipeline
Same parameters as used during training in `2.1-sleepstage_basemodels`.


In [ ]:
eeg_channels = ["eeg_c4", "eeg_f4", "eeg_o2", "eeg_fp1", "eeg_t3", "eeg_cz"]

lag_features = [
    "eeg_c4_bp_delta",
    "eeg_o2_bp_theta",
    "eog_e2_bp_slow",
    "eeg_cz_bp_delta",
    "eeg_cz_bp_high_gamma",
    "hr_mean",
    "temp_mean",
]
lag_shifts = (1, 2, 3)

feat_pipeline = SleepFeaturePipeline(
    ratio_cols=eeg_channels,
    lag_features=lag_features,
    lags=lag_shifts,
    original_cols=FEAT_COLS + FEAT_METADATA_COLS,
)

X_train, y_train, info_train = feat_pipeline.fit_transform(X_train, y_train, info_train)
X_val,   y_val,   info_val   = feat_pipeline.transform(X_val,   y_val,   info_val)
X_test,  y_test,  info_test  = feat_pipeline.transform(X_test,  y_test,  info_test)

feature_names = feat_pipeline.feature_names_

print(f"After feature engineering — train: {X_train.shape}, val: {X_val.shape}, test: {X_test.shape}")
print(f"Total features: {len(feature_names)}")


## Load Saved Models
Load the best hyper-parameters saved from the Optuna tuning runs in `2.1-sleepstage_basemodels`.


In [ ]:
XGB_DIR = PROJECT_ROOT / "models" / "xgb_featured"
LGB_DIR = PROJECT_ROOT / "models" / "lgb_featured"

# ── XGBoost ───────────────────────────────────────────────────────────────────
with open(XGB_DIR / "metadata.json") as f:
    xgb_meta = json.load(f)

xgb_best_params = xgb_meta["best_params"]
print(f"XGBoost best CV F1: {xgb_meta['best_cv_score']:.4f}")
print(f"XGBoost params: {xgb_best_params}")

# ── LightGBM ──────────────────────────────────────────────────────────────────
with open(LGB_DIR / "metadata.json") as f:
    lgb_meta = json.load(f)

lgb_best_params = lgb_meta["best_params"]
print(f"\nLightGBM best CV F1: {lgb_meta['best_cv_score']:.4f}")
print(f"LightGBM params: {lgb_best_params}")


## Retrain on Train + Val, Predict on Test
Combine train and validation sets so the models see more data before final evaluation.


In [ ]:
# Combine train + val
X_trainval  = np.concatenate([X_train, X_val],  axis=0)
y_trainval  = np.concatenate([y_train, y_val],  axis=0)
info_trainval = pd.concat([info_train, info_val], ignore_index=True)

w_trainval = compute_sample_weight(class_weight="balanced", y=y_trainval)

print(f"Train+Val size: {X_trainval.shape}  |  Test size: {X_test.shape}")

# ── XGBoost ───────────────────────────────────────────────────────────────────
xgb_model = XGBClassifier(
    objective="multi:softprob",
    num_class=5,
    eval_metric="mlogloss",
    tree_method="hist",
    random_state=42,
    n_jobs=-1,
    **xgb_best_params,
)
xgb_model.fit(X_trainval, y_trainval, sample_weight=w_trainval)
y_pred_xgb = xgb_model.predict(X_test)
print("XGBoost fitted.")

# ── LightGBM ──────────────────────────────────────────────────────────────────
lgb_model = LGBMClassifier(
    objective="multiclass",
    num_class=5,
    metric="multi_logloss",
    random_state=42,
    n_jobs=-1,
    verbosity=-1,
    **lgb_best_params,
)
lgb_model.fit(X_trainval, y_trainval, sample_weight=w_trainval)
y_pred_lgb = lgb_model.predict(X_test)
print("LightGBM fitted.")


## Test Set Evaluation


In [ ]:
STAGE_NAMES = ["W", "N1", "N2", "N3", "R"]

# ── Ensemble (equal-weight probability average) ───────────────────────────────
y_prob_xgb = xgb_model.predict_proba(X_test)
y_prob_lgb = lgb_model.predict_proba(X_test)
y_pred_ens = np.argmax(y_prob_xgb * 0.5 + y_prob_lgb * 0.5, axis=1)

for name, y_pred in [("XGBoost", y_pred_xgb), ("LightGBM", y_pred_lgb), ("Ensemble", y_pred_ens)]:
    f1 = f1_score(y_test, y_pred, average="macro")
    print(f"{'─'*50}")
    print(f"{name}  |  Test macro F1: {f1:.4f}")
    print(classification_report(y_test, y_pred, digits=4, target_names=STAGE_NAMES))


In [ ]:
# Confusion matrices
for name, y_pred in [("XGBoost", y_pred_xgb), ("LightGBM", y_pred_lgb), ("Ensemble", y_pred_ens)]:
    plot_confusion_matrix(
        {"confusion_matrix": confusion_matrix(y_test, y_pred)},
        STAGE_MAP,
        title=f"{name} — Test Set",
    )
    plt.show()


In [ ]:
# Per-subject macro F1 distributions
fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)
fig.suptitle("Per-subject macro F1 — Test Set")

for ax, (name, y_pred) in zip(axes, [("XGBoost", y_pred_xgb), ("LightGBM", y_pred_lgb), ("Ensemble", y_pred_ens)]):
    test_df = info_test.copy()
    test_df["y_true"] = y_test
    test_df["y_pred"] = y_pred
    scores = [
        f1_score(g.y_true, g.y_pred, average="macro")
        for _, g in test_df.groupby("subject_id")
    ]
    ax.hist(scores, bins=15, alpha=0.75, color="steelblue", edgecolor="white")
    ax.axvline(np.mean(scores), color="tomato", linestyle="--", label=f"mean={np.mean(scores):.3f}")
    ax.set_title(name)
    ax.set_xlabel("Macro F1")
    ax.legend()

axes[0].set_ylabel("Count")
plt.tight_layout()
plt.show()


In [ ]:
# Per-subject F1 breakdown table (ensemble)
test_df = info_test.copy()
test_df["y_true"] = y_test
test_df["y_pred"] = y_pred_ens

rows = []
for sid, g in test_df.groupby("subject_id"):
    report = classification_report(
        g.y_true, g.y_pred,
        labels=list(range(5)),
        target_names=STAGE_NAMES,
        output_dict=True,
        zero_division=0,
    )
    row = {"subject": sid, "macro_f1": report["macro avg"]["f1-score"]}
    for stage in STAGE_NAMES:
        row[f"f1_{stage}"] = report[stage]["f1-score"]
    rows.append(row)

subject_results = pd.DataFrame(rows).set_index("subject").sort_values("macro_f1", ascending=False)
print("Per-subject F1 (Ensemble) — Test Set")
subject_results.round(4)


## Post-processing: Mode Filter Smoothing
Apply a majority-vote window along the predicted sequence and compare against raw predictions.


In [ ]:
KERNEL_SIZE = 5  # must be odd; try 3, 5, 7

y_pred_xgb_smooth = mode_filter_1d(y_pred_xgb, kernel_size=KERNEL_SIZE)
y_pred_lgb_smooth = mode_filter_1d(y_pred_lgb, kernel_size=KERNEL_SIZE)
y_pred_ens_smooth = mode_filter_1d(y_pred_ens, kernel_size=KERNEL_SIZE)

comparisons = [
    ("XGBoost",         y_pred_xgb),
    (f"XGBoost+mode{KERNEL_SIZE}", y_pred_xgb_smooth),
    ("LightGBM",        y_pred_lgb),
    (f"LightGBM+mode{KERNEL_SIZE}", y_pred_lgb_smooth),
    ("Ensemble",        y_pred_ens),
    (f"Ensemble+mode{KERNEL_SIZE}", y_pred_ens_smooth),
]

for name, y_pred in comparisons:
    f1 = f1_score(y_test, y_pred, average="macro")
    print(f"{'─'*55}")
    print(f"{name:<30}  Test macro F1: {f1:.4f}")
    print(classification_report(y_test, y_pred, digits=4, target_names=STAGE_NAMES))


In [ ]:
# Confusion matrices: raw vs smoothed for each model
for model_name, y_raw, y_smooth in [
    ("XGBoost",  y_pred_xgb, y_pred_xgb_smooth),
    ("LightGBM", y_pred_lgb, y_pred_lgb_smooth),
    ("Ensemble", y_pred_ens, y_pred_ens_smooth),
]:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle(f"{model_name} — Raw vs Mode-filtered (k={KERNEL_SIZE})")
    for ax, y_pred, subtitle in zip(axes, [y_raw, y_smooth], ["Raw", f"Mode-filtered (k={KERNEL_SIZE})"]):
        cm = confusion_matrix(y_test, y_pred)
        cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True).clip(min=1)
        im = ax.imshow(cm_norm, cmap="Blues", vmin=0, vmax=1)
        ax.set_title(f"{subtitle}  |  F1={f1_score(y_test, y_pred, average='macro'):.4f}")
        ax.set_xticks(range(5)); ax.set_xticklabels(STAGE_NAMES)
        ax.set_yticks(range(5)); ax.set_yticklabels(STAGE_NAMES)
        ax.set_xlabel("Predicted"); ax.set_ylabel("True")
        for i in range(5):
            for j in range(5):
                ax.text(j, i, f"{cm_norm[i,j]:.2f}", ha="center", va="center",
                        color="white" if cm_norm[i,j] > 0.5 else "black", fontsize=8)
        fig.colorbar(im, ax=ax)
    plt.tight_layout()
    plt.show()
